# Notebook 14 — Offline Evaluation & Temporal Cohort Replay

**Purpose:** audit the completed Notebook 13 cascade using **frozen** August-selected settings and September test predictions. No model tuning on September. This is an **offline snapshot replay / future-label evaluation**, **not** an online event-by-event replay: September recommendations are fixed at the September 1 history cutoff and compared with future user views. No impressions, auction outcomes, or calibrated CTR are available.

Outputs: `reports/notebook14_offline_evaluation.json`, segment/cohort CSVs, and bootstrap paired-difference CSV. Uses precomputed scores and predictions; no retraining.

## 1. Setup, artifact checks, and frozen decisions

In [1]:
# pip install pandas numpy scipy
from pathlib import Path
import json, numpy as np, pandas as pd
ROOT=Path.cwd().resolve()
if not (ROOT/'data'/'processed').is_dir(): ROOT=ROOT.parent
assert (ROOT/'data'/'processed').is_dir(), 'Run from project root or notebooks/'
PROC=ROOT/'data'/'processed'; REPORTS=ROOT/'reports'; REPORTS.mkdir(exist_ok=True)
SEED=42; K=10
r13=json.loads((REPORTS/'notebook13_cascaded_pipeline.json').read_text())
r12=json.loads((REPORTS/'notebook12_mmr_reranking.json').read_text())
assert r13['selected_top_n']==20, 'Check Notebook 13 selected gate'
assert abs(r13['mmr_lambda']-r12['selected_lambda'])<1e-12
print('Frozen TopN',r13['selected_top_n'],'MMR lambda',r13['mmr_lambda'])


Frozen TopN 20 MMR lambda 0.9


## 2. Load recommendations and future labels; verify no leakage and identity

In [2]:
def load_predictions():
    paths={
        'hybrid':PROC/'notebook08_test_hybrid_candidates.csv',
        'lightgbm':PROC/'notebook09_test_scored_candidates.csv',
        'deepfm':PROC/'notebook10_test_scored_candidates.csv',
        'cascade':PROC/'notebook13_test_cascade_top10.csv',
    }
    for name,p in paths.items(): assert p.exists(),f'Missing {name}: {p}'
    raw={k:pd.read_csv(v) for k,v in paths.items()}
    for name,df in raw.items():
        assert {'visitorid','itemid'}.issubset(df.columns),f'{name}: {df.columns.tolist()}'
        assert not df[['visitorid','itemid']].duplicated().any(),name
    assert 'rank' in raw['hybrid'].columns
    assert 'new_rank' in raw['lightgbm'].columns and 'new_rank' in raw['deepfm'].columns
    print('Cascade columns:',raw['cascade'].columns.tolist())
    return raw
raw=load_predictions()
print('Cascade example:',raw['cascade'].head(2).to_dict('records'))


Cascade columns: ['visitorid', 'rank', 'itemid', 'lgb_score', 'lgb_rank', 'deepfm_score', 'deepfm_rank', 'fine_rank', 'final_rank']
Cascade example: [{'visitorid': 155, 'rank': 1, 'itemid': 269955, 'lgb_score': 0.03497876, 'lgb_rank': 3, 'deepfm_score': 2.351403, 'deepfm_rank': 1, 'fine_rank': 1, 'final_rank': 1}, {'visitorid': 155, 'rank': 17, 'itemid': 418277, 'lgb_score': 0.06166826, 'lgb_rank': 1, 'deepfm_score': 2.1993115, 'deepfm_rank': 2, 'fine_rank': 2, 'final_rank': 2}]


The cascade artifact contains the final Top-10 rows. We use its stored ordering if available; otherwise it must contain exactly 10 rows per user and already be ordered. A sorted `final_rank` is preferred; check the displayed columns before continuing.

In [3]:
c=raw['cascade'].copy()
rank_cols=[x for x in ('mmr_rank','final_rank','rank_mmr','rank','new_rank') if x in c.columns]
if rank_cols:
    cascade_rank_col=rank_cols[0]
    c=c.sort_values(['visitorid',cascade_rank_col,'itemid'],kind='stable')
else:
    raise ValueError('Cascade file has no rank column. Inspect columns and add its final rank name to rank_cols.')
assert c.groupby('visitorid').size().eq(K).all(), 'Cascade should contain exactly 10 rows/user'
assert c.groupby('visitorid')[cascade_rank_col].nunique().eq(K).all()
users=np.sort(c.visitorid.unique())
for name in ('hybrid','lightgbm','deepfm'):
    assert set(raw[name].visitorid.unique())==set(users),f'User mismatch: {name}'
    assert raw[name].groupby('visitorid').size().eq(100).all(),name

def topk(df,col,k=10):
    x=df.sort_values(['visitorid',col,'itemid'],kind='stable').groupby('visitorid',sort=False).head(k)
    return x[['visitorid','itemid']].copy()
pred={
 'hybrid':topk(raw['hybrid'],'rank'),
 'lightgbm':topk(raw['lightgbm'],'new_rank'),
 'deepfm':topk(raw['deepfm'],'new_rank'),
 'cascade':c[['visitorid','itemid']].copy()
}
for name,p in pred.items():
    assert len(p)==len(users)*K,(name,len(p))
    assert not p[['visitorid','itemid']].duplicated().any(),name
labels_path=PROC/'test_view_labels_warm_users.csv'
assert labels_path.exists(),labels_path
labels=pd.read_csv(labels_path,usecols=['visitorid','itemid']).drop_duplicates()
assert set(labels.visitorid.unique())==set(users)
labels['positive']=1
label_counts=labels.groupby('visitorid').size().rename('n_labels')
# Train/test cutoff guard; labels must come from the future-label export, not history.
hist=pd.read_csv(PROC/'test_history.csv',usecols=['timestamp','visitorid','itemid','event'])
cutoff_ms=int(pd.Timestamp('2015-09-01',tz='UTC').timestamp()*1000)
assert hist.timestamp.max()<cutoff_ms
seen=set(map(tuple,hist.loc[hist.event.eq('view'),['visitorid','itemid']].itertuples(index=False,name=None)))
for name,p in pred.items():
    assert not any(pair in seen for pair in p.itertuples(index=False,name=None)),f'Seen item in {name}'
print('Test users:',len(users),'future positive pairs:',len(labels),'all predictions validated')


Test users: 15137 future positive pairs: 31876 all predictions validated


## 3. Per-user paired evaluation: Recall@10 and HitRate@10

In [4]:
def per_user_hits(p,labels,users):
    joined=p.merge(labels,on=['visitorid','itemid'],how='left',validate='one_to_one')
    h=joined.assign(positive=joined.positive.fillna(0)).groupby('visitorid').positive.sum()
    return pd.Series(0.,index=users).add(h,fill_value=0).reindex(users).astype(float)
user_metrics=pd.DataFrame({'visitorid':users}).set_index('visitorid')
user_metrics['n_labels']=label_counts.reindex(users).astype(int)
for name,p in pred.items():
    user_metrics[name+'_hits']=per_user_hits(p,labels,users)
    user_metrics[name+'_recall']=user_metrics[name+'_hits']/user_metrics.n_labels
    user_metrics[name+'_hit_rate']=(user_metrics[name+'_hits']>0).astype(float)
summary=[]
for name in pred:
    h=user_metrics[name+'_hits']
    summary.append({'model':name,'Recall_micro':float(h.sum()/user_metrics.n_labels.sum()),
                    'Recall_macro':float(user_metrics[name+'_recall'].mean()),
                    'HitRate':float(user_metrics[name+'_hit_rate'].mean()),
                    'hits':int(h.sum()),
                    'CatalogCoverage_at_10':float(pred[name].itemid.nunique()/hist.itemid.nunique())})
summary=pd.DataFrame(summary)
print(summary.to_string(index=False))
expected=r13['test']['gated_deepfm_mmr']['Recall_micro']
actual=float(summary.loc[summary.model.eq('cascade'),'Recall_micro'].iloc[0])
assert np.isclose(actual,expected,atol=1e-10),(actual,expected)


   model  Recall_micro  Recall_macro  HitRate  hits  CatalogCoverage_at_10
  hybrid      0.020768      0.024067 0.037590   662               0.198016
lightgbm      0.026383      0.030129 0.047632   841               0.149187
  deepfm      0.027105      0.030666 0.048490   864               0.145178
 cascade      0.027387      0.030902 0.049019   873               0.147348


## 4. Paired user bootstrap (uncertainty, not significance proof)

Resample **users**, preserving each user's positive-label denominator and paired predictions. Micro recall difference is computed from the summed hits / summed labels in each resample. Confidence intervals reflect sampling variation under this fixed cohort; repeated test tuning would invalidate confirmatory claims.

In [5]:
rng=np.random.default_rng(SEED);B=2000
n=len(user_metrics)
counts=user_metrics.n_labels.to_numpy(dtype=np.float64)
comparisons=[('cascade','deepfm'),('cascade','lightgbm'),('cascade','hybrid')]
rows=[]
for a,b in comparisons:
    ah=user_metrics[a+'_hits'].to_numpy(dtype=np.float64)
    bh=user_metrics[b+'_hits'].to_numpy(dtype=np.float64)
    delta=(ah.sum()-bh.sum())/counts.sum()
    samples=np.empty(B)
    for j in range(B):
        idx=rng.integers(0,n,size=n)
        samples[j]=(ah[idx].sum()-bh[idx].sum())/counts[idx].sum()
    lo,hi=np.quantile(samples,[.025,.975])
    rows.append({'comparison':a+' minus '+b,'delta_micro_recall':float(delta),
                 'ci95_low':float(lo),'ci95_high':float(hi),
                 'bootstrap_fraction_positive':float((samples>0).mean()),'bootstrap_replicates':B})
bootstrap=pd.DataFrame(rows)
print(bootstrap.to_string(index=False))


            comparison  delta_micro_recall  ci95_low  ci95_high  bootstrap_fraction_positive  bootstrap_replicates
  cascade minus deepfm            0.000282 -0.000252   0.000793                       0.8425                  2000
cascade minus lightgbm            0.001004  0.000089   0.001889                       0.9800                  2000
  cascade minus hybrid            0.006619  0.005249   0.008130                       1.0000                  2000


## 5. User activity / sparsity cohorts

Segments use **only September 1 historical views**. Cohort analysis is descriptive and does not change model selection.

In [6]:
view_counts=hist.loc[hist.event.eq('view')].groupby('visitorid').size()
user_metrics['history_views']=view_counts.reindex(users).fillna(0).astype(int)
user_metrics['activity_cohort']=pd.cut(user_metrics.history_views,bins=[-1,1,3,10,np.inf],
                                       labels=['0–1 views','2–3 views','4–10 views','11+ views'])
segments=[]
for cohort,g in user_metrics.groupby('activity_cohort',observed=True):
    d={'cohort':str(cohort),'users':len(g),'future_positive_pairs':int(g.n_labels.sum())}
    for name in pred:
        d[name+'_recall_micro']=float(g[name+'_hits'].sum()/g.n_labels.sum())
        d[name+'_hit_rate']=float(g[name+'_hit_rate'].mean())
    segments.append(d)
segments=pd.DataFrame(segments)
print(segments.to_string(index=False))


    cohort  users  future_positive_pairs  hybrid_recall_micro  hybrid_hit_rate  lightgbm_recall_micro  lightgbm_hit_rate  deepfm_recall_micro  deepfm_hit_rate  cascade_recall_micro  cascade_hit_rate
 0–1 views   6984                   9862             0.026465         0.032932               0.033360           0.041237             0.033462         0.041667              0.033867          0.042383
 2–3 views   4067                   6301             0.028884         0.039095               0.036185           0.047455             0.037137         0.049176              0.037296          0.049176
4–10 views   2802                   5245             0.026120         0.039971               0.035081           0.054961             0.034700         0.052463              0.034700          0.052819
 11+ views   1284                  10468             0.007833         0.052960               0.009553           0.066978             0.011272         0.074766              0.011655          0.076324


## 6. Calendar-based future-label cohorts (fixed-cutoff replay)

Each future label is assigned to a day/week **after September 1**. Predictions stay fixed throughout the period. This is *not* rolling retraining or a true real-time replay. Use only the original test future events, filter `view`, and join to the same warm-user label universe.

In [7]:
future=pd.read_csv(PROC/'test_future_events.csv',usecols=['timestamp','visitorid','itemid','event'])
future=future.loc[future.event.eq('view'),['timestamp','visitorid','itemid']].copy()
assert future.timestamp.min()>=cutoff_ms
future=future.merge(labels[['visitorid','itemid']],on=['visitorid','itemid'],how='inner')
future['day']=pd.to_datetime(future.timestamp,unit='ms',utc=True).dt.strftime('%Y-%m-%d')
# Deduplicate each user-item across entire future horizon so each positive is counted once.
future=future.sort_values('timestamp').drop_duplicates(['visitorid','itemid'],keep='first')
assert len(future)==len(labels),'Label universe must match first future view events'
future['week_start']=pd.to_datetime(future.timestamp,unit='ms',utc=True).dt.to_period('W').dt.start_time.dt.strftime('%Y-%m-%d')
weekly=[]
for wk,g in future.groupby('week_start',sort=True):
    row={'week_start':wk,'positive_pairs':len(g),'users_with_future_views':g.visitorid.nunique()}
    keys=g[['visitorid','itemid']]
    for name,p in pred.items():
        hits=len(keys.merge(p,on=['visitorid','itemid'],how='inner'))
        row[name+'_hits']=hits
        row[name+'_recall_micro']=float(hits/len(g))
    weekly.append(row)
weekly=pd.DataFrame(weekly)
print(weekly.to_string(index=False))


C:\Users\wyr95\AppData\Local\Temp\ipykernel_8812\2323377160.py:9: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  future['week_start']=pd.to_datetime(future.timestamp,unit='ms',utc=True).dt.to_period('W').dt.start_time.dt.strftime('%Y-%m-%d')


week_start  positive_pairs  users_with_future_views  hybrid_hits  hybrid_recall_micro  lightgbm_hits  lightgbm_recall_micro  deepfm_hits  deepfm_recall_micro  cascade_hits  cascade_recall_micro
2015-08-31           14396                     7359          364             0.025285            455               0.031606          473             0.032856           474              0.032926
2015-09-07           15492                     7720          261             0.016847            344               0.022205          348             0.022463           355              0.022915
2015-09-14            1988                     1140           37             0.018612             42               0.021127           43             0.021630            44              0.022133


## 7. Export reproducible artifacts and interpretation

In [8]:
summary.to_csv(REPORTS/'notebook14_model_summary.csv',index=False)
bootstrap.to_csv(REPORTS/'notebook14_paired_bootstrap.csv',index=False)
segments.to_csv(REPORTS/'notebook14_activity_segments.csv',index=False)
weekly.to_csv(REPORTS/'notebook14_future_weekly_cohorts.csv',index=False)
report={
 'method':'frozen-cutoff offline snapshot replay and paired bootstrap',
 'protocol':'September 1 history cutoff; future view labels; Notebook13 parameters frozen from August',
 'selected_top_n':r13['selected_top_n'],'mmr_lambda':r13['mmr_lambda'],
 'test_users':int(len(users)),'test_positive_pairs':int(len(labels)),
 'model_summary':summary.to_dict('records'),
 'paired_user_bootstrap':bootstrap.to_dict('records'),
 'activity_segments':segments.to_dict('records'),
 'fixed_snapshot_weekly_future_cohorts':weekly.to_dict('records'),
 'caveats':['RetailRocket views are not ad impressions or observed non-clicks',
            'No causal uplift or calibrated CTR/CVR can be inferred',
            'Future calendar cohorts reuse a single September 1 recommendation snapshot, not online serving',
            'Bootstrap intervals are descriptive and conditional on this offline evaluation cohort',
            'DeepFM stage scores were precomputed on Top100 in Notebook13; no actual conditional inference latency measured']}
out=REPORTS/'notebook14_offline_evaluation.json'
out.write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
print('Saved:',out)


Saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\reports\notebook14_offline_evaluation.json


### What to report

- Primary: final cascade **Recall@10**, **HitRate@10**, and catalog coverage vs baselines.
- Uncertainty: paired bootstrap CI for differences, without claiming a statistically established win if CI crosses zero.
- Robustness: sparse-history user cohorts and fixed-cutoff future weeks.
- Engineering next: **Notebook 15** measures actual model inference and candidate-gating latency; do not use the precomputed replay time as serving p95.